In [87]:
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import numpy.typing as npt

DATA = Path("/home/seba/Projects/NoTorchGrad/datasets/encoded_text.npy")
ids = np.load(DATA)

In [58]:
class Dataset:

    @staticmethod
    def batches(ids, batch_size: int =16, context_length: int = 8): 
        """Define the inputs and target batches for training.
        Simple version always shuffles."""
        X = [] # inputs
        y = [] # targets
        # want to make sure that the len of ids is divisible by 64 for gpu usage (not that applicable here but nice to implement)
        leftover = (len(ids) - (len(ids)%64))
        ids = ids[:leftover]
        idx = 0
        while idx + context_length < len(ids):
            # safegaurd to make sure we don't go over the amount of ids
            if idx + context_length >= len(ids):
                break
            X.append(ids[idx:idx+context_length])
            y.append(ids[idx+1:idx+context_length+1])
            idx += context_length
        # need to set X and y as arrays
        X = np.asarray(X)
        y = np.asarray(y)
        # shuffle all the sequences
        shuffle = np.random.permutation(len(X))
        X = X[shuffle]
        y = y[shuffle]
        # make the num of batches you need
        num_batches = len(X)//batch_size
        # shape X and y into sequences to get rid of any excess items that won't make it into the training run and would mess up our math
        X = X[:int(num_batches)*int(batch_size)]
        y = y[:int(num_batches)*int(batch_size)]
        # reshape into batches for training
        # we don't know how many batches there will be, so we will define that dimension as -1
        X = X.reshape(-1, batch_size, context_length)
        y = y.reshape(-1, batch_size, context_length)
        
        return X, y

    @staticmethod
    def __len__(X):
        """Returns the length of inputs."""
        return len(X)

    @staticmethod
    def unique_vocab(ids):
        """Returns the number of unique vocab."""
        return len(ids)


In [86]:
X, y = Dataset.batches(ids=ids)

In [8]:
class Math:
    "Contains the algorithms I will need to perform throughout the layers."
    "Z correlates to the output of the previous calculation."
    

    @staticmethod
    def ReLU(Z):
        """Return the activations of the previous dot product."""
        A = np.maximum(0, Z)
        return A

    @staticmethod
    def softmax(Z):
        """Takes the output from the last layer and transforms
        each vector into a probability distribution adding to 1.
        REMEMBER!! loop through each vector in the batch."""
        for vector in Z:
            # log of each dimension in the vector
            num = np.log(vector)
            # sum of the logs
            den = np.sum(num)
            softmax = num/den
            return softmax # sum of vector values = 1



In [105]:
class EmbeddingsLayer:
    """Feed your inputs into the embeddings layer, output the inputs as embedded tokens.
    There are 1000 unique tokens, so that will be the n_rows which defaults to 1000."""
    # just need the len of vocab
    def __init__(self, n_tokens:int = 1000, n_dims: int = 100):
        self.embeddings_layer = np.random.randn(n_tokens, n_dims)

    def embed_tokens(self, tokens: npt.NDArray) -> npt.NDArray:
        """Take in a list of token inputs and replace with their embedded vector."""
        embed_tokens = [self.embeddings_layer[t] for t in tokens]
        self.embedded_tokens = np.asarray(embed_tokens)

        return self.embedded_tokens.T

    def __len__(self,):
        """Returns the number of token inputs coming through the embeddings layer."""
        return len(self.embedded_tokens)

    def __ndims__(self):
        return self.embeddings_layer.shape[1]



In [ ]:
class HiddenLayers:
    """Define your layers here where n_inputs = number of rows
    and n_dims = number of dimensions in each vector.
    In this case I will predefine n_dims because I predefined that in the embeddings layer,
    n_dims = 100"""
    # define layers to pass through
    def __init__(self, n_inputs: int, n_dims: int):
        # transposing this because we are sending (embed_rows x 100) into the layers
        # (emb_rows x 100)*(n_inputs x 100) no work, but (embed_rows x 100)*(100 x n_inputs does)
        # keep small by making weights 10% original size
        self.w = np.random.randn(n_inputs, n_dims) # w for weights
        # instantialize the bias to start off at zeros w/ size
        # of columns after the dot product
        self.bias = np.zeros((1, n_dims))
        self.activation_func = Math.ReLU

    def forward(self, inputs: npt.NDArray):
        """Activation function wrapped around inputs*weights -> A(X*W)
        Returns: 
            Matrix after activation function."""
        z = np.dot(inputs, self.w) + self.bias # z for output
        self.a = self.activation_func(z) # a for activation
        # define num inputs "_len" coming from this layer into the next
        self.n_dims = self.a.shape[1]
        return self.a, 

    def n_dims(self,):
        """Returns:
            number of dimensions coming from previous layer."""
        return self.a.shape[1]

    def n_outputs(self,):
        return self.a.shape[0]
    

In [ ]:
# test tokens 
tokens = np.array([1,2,3,4])
# instantialize embeddings layer
emb_layer = EmbeddingsLayer()
emb_tokens = emb_layer.embed_tokens(tokens=tokens)
emb_len = emb_layer.__len__()
emb_dims = emb_layer.__ndims__()

def_H = HiddenLayers
# layer one takes in the embeddings inputs
h_layer1 = def_H(emb_len, emb_dims)
a1 = h_layer1.forward(emb_tokens)

h_layer2 = def_H(n_outputs1, n_dims1)
a2, n_dims2 = h_layer2.forward(a1)
a2



array([[ 0.        , 11.80942801,  0.        , ...,  3.72052915,
         0.        ,  8.45308934],
       [17.13064878,  0.        ,  2.43550568, ...,  0.        ,
        10.09610161,  5.25154389],
       [ 0.        ,  4.25229417,  0.92814845, ...,  0.        ,
         4.00508062,  9.66204119],
       ...,
       [ 0.        ,  5.9669974 ,  0.75108743, ...,  0.        ,
        24.97446246, 33.19483499],
       [ 0.        ,  0.        ,  2.35588221, ...,  0.        ,
        12.53399111, 14.06752847],
       [ 0.        ,  0.        ,  0.        , ...,  0.        ,
        17.38999884, 25.88088587]], shape=(100, 100))

In [ ]:
class Train:
    """This class will automatically take your input tokens and run them through
    their embeddings and the first layer"""
    def __init__(self,) -> None:
        pass